# Project 2 — Word embeddings and sequence models

**Question:** do pretrained word vectors and word order help classify WildGuardMix prompts?

Read and run Q1–Q6 in order. Each code cell names the question it addresses.
Each section ends with an answer based on its outputs, plus what remains uncertain.
Q3 includes one matched Project 1 reference baseline on the same fold.

We reuse the course embedding loader, encoders, classifier, batching and training loop.
Repeated project operations are in [`utils/p2/functions.py`](../utils/p2/functions.py).
The official questions remain in [`project.ipynb`](project.ipynb).

## Setup and experiment budget

Run with the repository `.venv` kernel. The device is detected at runtime.
This notebook runs a **development pilot on fold 0**, with ten epochs per model.
It is not a completed five-fold experiment. Change the settings here to experiment.
The official test is not loaded during model selection.

In [ ]:
# Setup — imports, existing HF login and hardware
import json
import os
import platform
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "part2/clslib.py").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Keep the normal Hugging Face login location if notebook caches are redirected.
default_hf_home = Path(os.environ.get("HF_HOME") or
                       Path(os.environ.get("XDG_CACHE_HOME") or Path.home() / ".cache") / "huggingface")
os.environ.setdefault("HF_TOKEN_PATH", str(default_hf_home / "token"))

from datasets import load_dataset
import matplotlib.pyplot as plt
from importlib.metadata import version
from sklearn.linear_model import LogisticRegression
from part1 import experiment as p1_experiment
from part2 import clslib, experiment
from utils.p1.functions import TFIDF as P1TFIDF, encode_texts as p1_encode_texts
from utils.p2.functions import (
    ProjectWordTokenizer, balanced_sample, classification_scores, clean_wildguard_prompts,
    coverage_by_label, embedding_coverage, make_toy_embeddings, stratified_folds,
    nearest_words, prepare_fold, run_configuration, input_features,
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Setup — change these values to experiment
use_toy_data = os.environ.get("P2_USE_TOY_DATA", "0") == "1"
seed = 42
examples_per_class = None  # Derived from the cleaned class counts below.
num_folds = 5
fold_index = 0
embedding_dims = 50
num_epochs = 10  # Actually used for every fit below; start with a small pilot.
batch_size = 64
hidden_size = 64
dropout = 0.2
learning_rate = 0.001
weight_decay = 0.0001
repeat_seeds = [seed, seed + 100, seed + 200, seed + 300, seed + 400]  # Initialization/batch-order check on the SAME data.
dataset_revision = "d29c47f41c8b51348b5c8e8c81c039b3132b66d1"
glove_path = Path(os.environ.get("P2_GLOVE_PATH", root / "part2/glove.2024.wikigiga.50d.zip"))
dataset_cache = Path(os.environ.get("P2_HF_DATASETS_CACHE", root / ".cache/huggingface/datasets"))
results_root = Path(os.environ.get("P2_RESULTS_ROOT", root / "part2/results"))
snellius_path = os.environ.get("P2_SNELLIUS_RESULTS")
snellius_results = Path(snellius_path) if snellius_path else None
require_cuda = os.environ.get("P2_REQUIRE_CUDA", "0") == "1"
if require_cuda and device != "cuda":
    raise RuntimeError("P2_REQUIRE_CUDA=1 but CUDA is unavailable")

if use_toy_data:
    examples_per_class, num_folds, embedding_dims = 6, 3, 12
    num_epochs, batch_size, hidden_size = 2, 4, 8

run_id = os.environ.get("P2_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
results_folder = results_root / ("toy" if use_toy_data else "pilot") / run_id
results_folder.mkdir(parents=True, exist_ok=False)
settings = dict(status="SMOKE_NOT_SCIENTIFIC" if use_toy_data else "DEVELOPMENT_PILOT",
                sample_seed=seed, examples_per_class=examples_per_class, num_folds=num_folds,
                fold_index=fold_index, embedding_dims=embedding_dims, epochs=num_epochs,
                batch_size=batch_size, hidden_size=hidden_size, dropout=dropout,
                learning_rate=learning_rate, weight_decay=weight_decay,
                repeat_seeds=repeat_seeds, device=device, dataset_revision=dataset_revision,
                embedding_source=str(glove_path), dataset_cache=str(dataset_cache),
                official_test_accessed=False, require_cuda=require_cuda,
                checkpoint_rule="last epoch; no validation checkpoint selection",
                vocabulary="pretrained rows observed in training only; other tokens map to UNK",
                tokenizer="lowercase, punctuation removed, stopwords kept, Treebank",
                packages={name: version(name) for name in
                          ["numpy", "pandas", "torch", "nltk", "datasets", "scikit-learn"]})
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print(settings["status"], "|", num_epochs, "epochs per fit | outputs:", results_folder)

## Q1. Is the embedding appropriate for this dataset?

**Answer with:** the text/label definition, tokenization examples, token and type coverage,
coverage by label, and frequent missing tokens.

We start with the tutorial's English GloVe 2024 Wikipedia + Gigaword **50d** table.
English general text makes it a reasonable candidate; security jargon and typos need checking.
We lowercase, remove punctuation, and keep stopwords (including negation).
The helper extends the course tokenizer with resource-free Treebank tokenization.
This is an explicit approximation to GloVe's preprocessing, not a claim of an exact match.
Removing punctuation splits contractions and changes URLs/code; inspect the examples below.

In [ ]:
# Q1a — load, clean, balance and split the same task as Project 1
if use_toy_data:
    raw = pd.DataFrame([
        {"prompt": f"Please water the garden plants number {i}",
         "prompt_harm_label": "unharmful", "subcategory": "synthetic"}
        for i in range(8)
    ] + [
        {"prompt": f"Synthetic security misuse request number {i}",
         "prompt_harm_label": "harmful", "subcategory": "synthetic"}
        for i in range(8)
    ] + [
        {"prompt": "excluded example", "prompt_harm_label": "harmful",
         "subcategory": " toxic_language_hate_speech "},
        {"prompt": "duplicate safe prompt", "prompt_harm_label": "unharmful",
         "subcategory": "synthetic"},
        {"prompt": "duplicate safe prompt", "prompt_harm_label": "unharmful",
         "subcategory": "synthetic"},
        {"prompt": "contradiction", "prompt_harm_label": "harmful",
         "subcategory": "synthetic"},
        {"prompt": "contradiction", "prompt_harm_label": "unharmful",
         "subcategory": "synthetic"},
    ])
else:
    dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtrain", split="train",
        revision=dataset_revision, token=True,
        cache_dir=str(dataset_cache),
    )
    dataset_provenance = {"repository": "allenai/wildguardmix",
                          "configuration": "wildguardtrain", "split": "train",
                          "requested_revision": dataset_revision,
                          "builder_name": dataset.info.builder_name,
                          "config_name": dataset.info.config_name}
    raw = dataset.select_columns(
        ["prompt", "prompt_harm_label", "subcategory"]).to_pandas()

data, data_audit = clean_wildguard_prompts(raw)
display(pd.Series(data_audit, name="data audit"))

historical_counts = {"original_rows": 86759, "excluded_category_rows": 3020,
                     "eligible_rows": 83739, "retained_rows": 46692}
if not use_toy_data:
    display(pd.DataFrame({"current pinned revision": data_audit,
                          "historical reference": historical_counts}))

if examples_per_class is None:
    examples_per_class = int(data.groupby("label").size().min())
sample = balanced_sample(data, examples_per_class, seed)
excluded_by_balancing = len(data) - len(sample)
print(f"Balanced sample: {examples_per_class:,} per class; "
      f"{excluded_by_balancing:,} retained majority-class examples excluded.")
folds = stratified_folds(sample["label"], num_folds, seed)
train_indices, validation_indices = folds[fold_index]
train = sample.iloc[train_indices].reset_index(drop=True)
validation = sample.iloc[validation_indices].reset_index(drop=True)
print("First fold:", len(train), "training /", len(validation), "validation")
assert set(train.id).isdisjoint(validation.id)
split_table = sample[["id", "label"]].copy()
split_table["fold"] = -1
for fold, (_, indices) in enumerate(folds):
    split_table.loc[indices, "fold"] = fold
split_table.to_csv(results_folder / "folds.csv", index=False)
data_record = {**data_audit, "examples_per_class": examples_per_class,
               "excluded_by_balancing": excluded_by_balancing}
if not use_toy_data:
    data_record["provenance"] = dataset_provenance
settings["examples_per_class"] = examples_per_class
settings["retained_rows"] = len(data)
settings["sample_rows"] = len(sample)
(results_folder / "data_audit.json").write_text(json.dumps(data_record, indent=2))
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))

In [ ]:
# Q1b — load local vectors and inspect tokenization
if use_toy_data:
    embeddings = make_toy_embeddings(
        ["please", "water", "the", "garden", "plants", "synthetic", "security", "request",
         "number", "safe", "prompt", "dog", "bites", "man"], embedding_dims, seed)
else:
    if not glove_path.exists():
        raise FileNotFoundError(f"Local GloVe archive required: {glove_path}")
    embeddings = clslib.PretrainedEmbeddings(str(glove_path), num_dims=embedding_dims)
tokenizer = ProjectWordTokenizer(lowercase=True, remove_punct=True, remove_stopwords=False,
                                 vocab=embeddings.vocab)
examples = ["Don't share your MFA code.", "Explain authentication and phishing.",
            "Please help with my gardenn."]  # Safe examples, not dataset prompts.
display(pd.DataFrame({"text": examples, "tokens": [tokenizer.tokens(s) for s in examples]}))
print("Full embedding matrix:", embeddings.vectors.shape)
print("Vector array MiB:", round(embeddings.vectors.nbytes / 2**20, 1))

In [ ]:
# Q1c — training coverage, label-specific coverage, and OOV counts
coverage, oov = embedding_coverage(train.prompt, tokenizer, embeddings.vocab)
label_coverage = coverage_by_label(train, tokenizer, embeddings.vocab)
display(coverage)
display(label_coverage.round(4))
display(oov.head(15))
coverage.to_csv(results_folder / "embedding_coverage.csv")
label_coverage.to_csv(results_folder / "coverage_by_label.csv", index=False)
oov.to_csv(results_folder / "oov_counts.csv", index=False)
print(f"Q1 answer: the table covers {coverage.token_coverage:.2%} of token occurrences "
      f"and {coverage.type_coverage:.2%} of distinct training tokens.")
print("Frequent-token coverage and rare-word coverage differ; coverage alone does not establish classification quality.")

## Q2. What does the embedding capture, and where does it fail?

**Answer with:** real neighbours for task vocabulary, an order-swap example,
and a controlled classification ablation in Q3.

GloVe assigns one vector per word, so it cannot give different representations to
different senses of the same word. Topic similarity can bring safe and unsafe requests
close together. Missing words collapse into one `<unk>` vector. Mean/sum pooling also
discards word order. These limitations matter when intent depends on context or negation.

In [ ]:
# Q2a — actual nearest neighbours; no invented similarities for missing words
probe_words = ["phishing", "fraud", "authentication", "mfa", "garden"]
display(pd.DataFrame({"word": probe_words, "in_vocabulary": [w in embeddings.vocab for w in probe_words]}))
neighbours = pd.concat([nearest_words(embeddings, word) for word in probe_words], ignore_index=True)
display(neighbours.round(4))
neighbours.to_csv(results_folder / "embedding_neighbours.csv", index=False)
print("These are distributional associations; inspect whether they express topic, syntax, or intent.")

In [ ]:
# Q2b — a concrete limitation: pooled vectors lose word order
sentences = ["dog bites man", "man bites dog"]
vectors = [np.mean([embeddings.embed_word(t) for t in tokenizer.tokens(s)], axis=0)
           for s in sentences]
distance = float(np.linalg.norm(vectors[0] - vectors[1]))
print("Mean-vector distance for the order swap:", distance)
assert np.allclose(vectors[0], vectors[1], atol=1e-6)
print("Q2 answer: the same words produce the same pooled vector, although the roles changed.")
print("Task-specific evidence for pretraining comes from Q3's trainable GloVe vs trainable random comparison.")
print("We have not compared another pretrained family or embedding dimension.")

## Q3. Which embed-and-pool choices matter?

**Answer with:** mean vs sum pooling, frozen vs trainable GloVe, and trainable
GloVe vs trainable random vectors. The last pair holds architecture, vocabulary,
trainability and budget fixed to examine initialization. A single pilot cannot prove an advantage.

Only training words covered by GloVe receive individual embedding rows; other
training and validation words map to a shared `<unk>` representation. Validation
never expands this training-only vocabulary. We report type and token coverage.
We use the course classifier (two tanh layers), AdamW and a fixed epoch budget.
The setting `num_epochs` controls the training loop. The final epoch is scored.
A matched BPE + TF-IDF + logistic Project 1 baseline is trained on this same fold.
That comparison tests whole systems; only tuned-versus-random isolates initialization.

In [ ]:
# Q3a — prepare shared inputs and explicit configurations
prepared = prepare_fold(train, validation, embeddings, tokenizer)
restricted_coverage, _ = embedding_coverage(validation.prompt, tokenizer, prepared["vocab"])
full_validation_coverage, _ = embedding_coverage(validation.prompt, tokenizer, embeddings.vocab)
display(pd.DataFrame({"full pretrained table": full_validation_coverage,
                      "training-only model vocabulary": restricted_coverage}))
print("Compact matrix:", prepared["vectors"].shape, "| same IDs for every configuration")
common = dict(hidden_size=hidden_size, classifier_width=hidden_size, dropout=dropout)
pool_configs = [
    dict(name="mean_frozen", encoder="mean", initialization="pretrained", frozen=True, **common),
    dict(name="sum_frozen", encoder="sum", initialization="pretrained", frozen=True, **common),
    dict(name="mean_tuned", encoder="mean", initialization="pretrained", frozen=False, **common),
    dict(name="mean_random", encoder="mean", initialization="random", frozen=False, **common),
]
fit_options = dict(epochs=num_epochs, batch_size=batch_size, learning_rate=learning_rate,
                   weight_decay=weight_decay, device=device)
score_rows, histories, prediction_tables = [], [], []

In [ ]:
# Q3b — run the four pooling comparisons on the same fold
for config in pool_configs:
    score, history, predictions = run_configuration(prepared, config, seed=seed, **fit_options)
    score_rows.append(score)
    histories.append(history)
    prediction_tables.append(predictions)
pool_results = pd.DataFrame(score_rows).set_index("name")
display(pool_results[["macro_f1", "harmful_recall", "benign_fpr", "trainable_parameters", "fit_seconds"]].round(4))
for a, b, question in [("mean_frozen", "sum_frozen", "sum versus mean"),
                       ("mean_frozen", "mean_tuned", "fine-tuning GloVe"),
                       ("mean_random", "mean_tuned", "pretraining at equal trainability")]:
    delta = pool_results.loc[b, "macro_f1"] - pool_results.loc[a, "macro_f1"]
    print(f"Q3 observed difference ({question}): {delta:+.4f} macro-F1 (B - A).")
print("These pilot differences are not yet stable estimates.")

# Tutorial 2 comparison: genuine per-epoch histories, with no best-epoch selection.
q3_curves = pd.concat(histories, ignore_index=True)
q3_curves = q3_curves[q3_curves.system.isin(["mean_tuned", "mean_random"])]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for system, curve in q3_curves.groupby("system"):
    curve = curve.sort_values("epoch")
    axes[0].plot(curve.epoch, curve.train_loss, marker="o", label=system)
    axes[1].plot(curve.epoch, curve.validation_loss, marker="o", label=system)
    axes[2].plot(curve.epoch, curve.macro_f1, marker="o", label=system)
for ax, title, ylabel in zip(axes, ["Training", "Validation", "Validation"],
                              ["NLL loss", "NLL loss", "Macro-F1"]):
    ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
    ax.legend()
fig.tight_layout()
fig.savefig(results_folder / "mean_pretrained_vs_random_curves.png", dpi=150)
plt.show()

# One matched Project 1 reference: fit every component on this training fold only.
p1_train_texts = train.prompt.tolist()
p1_validation_texts = validation.prompt.tolist()
p1_tokenizer = p1_experiment.train_tokenizer(
    p1_train_texts, vocab_size=3000, min_frequency=2)
p1_tfidf = P1TFIDF(p1_tokenizer).fit(p1_train_texts)
p1_x_train = p1_encode_texts(p1_train_texts, p1_tfidf)
p1_x_validation = p1_encode_texts(p1_validation_texts, p1_tfidf)
p1_model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=seed)
p1_model.fit(p1_x_train, train.label.to_numpy())
p1_predicted = p1_model.predict(p1_x_validation)
p1_probability = p1_model.predict_proba(p1_x_validation)
p1_score = {"name": "p1_bpe_tfidf_logistic", "seed": seed,
            "n_train": len(train), "n_validation": len(validation),
            **classification_scores(validation.label, p1_predicted)}
p1_predictions = validation[["id", "label"]].rename(columns={"label": "gold"}).copy()
p1_predictions["system"] = p1_score["name"]
p1_predictions["seed"] = seed
p1_predictions["prediction"] = p1_predicted
p1_predictions["probability_harmful"] = p1_probability[:, 1]
p1_predictions["confidence"] = p1_probability.max(axis=1)
comparison = pd.concat([
    pd.DataFrame([p1_score]),
    pd.DataFrame(score_rows).query("name in ['mean_tuned', 'mean_random']")
], ignore_index=True)
display(comparison[["name", "n_train", "macro_f1", "harmful_recall", "benign_fpr"]].round(4))
print("P1 comparison scope: same IDs, but a whole-system comparison rather than a pure embedding effect.")

## Q4. Does a recurrent encoder help?

**Answer with:** LSTM vs mean pooling, frozen vs trainable embeddings, half vs full
training data, and a dropout change. We keep validation fixed for the size comparison
and use a nested balanced subset of training examples. The half-data vocabulary is
rebuilt from that subset only, while the validation IDs remain identical.

Loss curves show optimization and possible overfitting. Rising validation loss while
training loss falls is a warning, not a cause diagnosis. This short budget does not
establish the best performance an LSTM can reach. An inner development split would
be needed for early stopping without selecting on the reported evaluation fold.

In [ ]:
# Q4a — recurrent architecture, trainability, data size and dropout
lstm_configs = [
    dict(name="lstm_frozen", encoder="lstm", initialization="pretrained", frozen=True, **common),
    dict(name="lstm_tuned", encoder="lstm", initialization="pretrained", frozen=False, **common),
    dict(name="lstm_half_data", encoder="lstm", initialization="pretrained", frozen=True, **common),
    dict(name="lstm_classifier_dropout", encoder="lstm", initialization="pretrained", frozen=True,
         **{**common, "dropout": 0.4}),
]
half_per_class = max(1, int(train.groupby("label").size().min() // 2))
small_train = balanced_sample(train, half_per_class, seed)
small_prepared = prepare_fold(small_train, validation, embeddings, tokenizer)
assert small_prepared["validation"].id.tolist() == prepared["validation"].id.tolist()
for config in lstm_configs:
    inputs = small_prepared if config["name"] == "lstm_half_data" else prepared
    score, history, predictions = run_configuration(inputs, config, seed=seed, **fit_options)
    score_rows.append(score)
    histories.append(history)
    prediction_tables.append(predictions)
all_results = pd.DataFrame(score_rows).set_index("name")
display(all_results.loc[["mean_frozen"] + [c["name"] for c in lstm_configs],
                        ["n_train", "macro_f1", "harmful_recall", "benign_fpr", "trainable_parameters"]].round(4))
for a, b in [("mean_frozen", "lstm_frozen"), ("lstm_frozen", "lstm_tuned"),
              ("lstm_half_data", "lstm_frozen"),
              ("lstm_frozen", "lstm_classifier_dropout")]:
    print(f"Q4: {b} minus {a}: {all_results.loc[b, 'macro_f1'] - all_results.loc[a, 'macro_f1']:+.4f} macro-F1.")

In [ ]:
# Q4b — inspect training and validation loss over epochs
history_table = pd.concat(histories, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, system in zip(axes, ["mean_frozen", "lstm_frozen"]):
    curve = history_table[history_table.system == system]
    ax.plot(curve.epoch, curve.train_loss, marker="o", label="training (with dropout)")
    ax.plot(curve.epoch, curve.validation_loss, marker="o", label="validation")
    ax.set(title=system, xlabel="Epoch", ylabel="NLL loss")
    ax.legend()
fig.tight_layout()
fig.savefig(results_folder / "learning_curves.png", dpi=150)
plt.show()
for system, curve in history_table.groupby("system"):
    curve = curve.sort_values("epoch")
    warning = ((curve.train_loss.diff() < 0) & (curve.validation_loss.diff() > 0)).sum()
    print(system, ":", int(warning), "epoch transitions with falling train / rising validation loss")
print("Q4 limit: training loss uses dropout and changing parameters; the curves are a diagnostic.")

## Q5. How much evidence supports our choices?

**Search plan:** eight motivated configurations, changing one factor at a time within
each contrast. Then repeat `mean_tuned` and `mean_random` across five seeds.
The sample and fold are fixed: this measures initialization/batch-order variation,
not dataset-selection or fold variation. Five seeds are still a pilot, not a final ranking.

After this pilot, choose a shortlist, lock settings and evaluate shared folds/seeds.
Report paired differences and descriptive spread. Do not divide a fold SD by the
square root of the number of folds as though they were independent measurements.
Do not select models on the official test.

**Separate historical evidence:** a completed Snellius study with a different evaluation
and vocabulary protocol reported about 0.8384 for pretrained mean pooling, 0.8385 for
random mean pooling, 0.8297 for pretrained LSTM, 0.8267 for LSTM with word dropout,
and 0.8076 for its matched P1 baseline. Those values are not pooled with this pilot.
The main lesson is uncertainty: pretraining was not consistently better there, even if
this development fold shows a small positive difference. Configure `P2_SNELLIUS_RESULTS`
to display the saved study files rather than treating these rounded values as fresh output.

In [ ]:
# Q5a — repeat the prespecified pretrained-versus-random comparison
repeat_configs = [pool_configs[2], pool_configs[3]]
for model_seed in repeat_seeds[1:]:
    for config in repeat_configs:
        score, history, predictions = run_configuration(prepared, config, seed=model_seed, **fit_options)
        score_rows.append(score)
        histories.append(history)
        prediction_tables.append(predictions)
scores = pd.DataFrame(score_rows)
history_table = pd.concat(histories, ignore_index=True)
predictions = pd.concat(prediction_tables, ignore_index=True)
scores["fold"] = fold_index
predictions["fold"] = fold_index
variance = scores[scores.name.isin(["mean_tuned", "mean_random"])].groupby("name").agg(
    mean_f1=("macro_f1", "mean"), seed_sd=("macro_f1", "std"), runs=("seed", "nunique"))
display(variance.round(4))
paired = scores[scores.name.isin(["mean_tuned", "mean_random"])].pivot(
    index="seed", columns="name", values="macro_f1")
paired["tuned_minus_random"] = paired.mean_tuned - paired.mean_random
display(paired.round(4))

repeated_curves = history_table[history_table.system.isin(["mean_tuned", "mean_random"])]
seed_curves = repeated_curves.groupby(["system", "epoch"]).agg(
    train_loss_mean=("train_loss", "mean"), train_loss_sd=("train_loss", "std"),
    validation_loss_mean=("validation_loss", "mean"), validation_loss_sd=("validation_loss", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std")).reset_index()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for system, curve in seed_curves.groupby("system"):
    for ax, mean_column, sd_column in zip(
            axes, ["train_loss_mean", "validation_loss_mean", "macro_f1_mean"],
            ["train_loss_sd", "validation_loss_sd", "macro_f1_sd"]):
        ax.plot(curve.epoch, curve[mean_column], marker="o", label=system)
        ax.fill_between(curve.epoch, curve[mean_column] - curve[sd_column],
                        curve[mean_column] + curve[sd_column], alpha=0.15)
for ax, title, ylabel in zip(axes, ["Training", "Validation", "Validation"],
                              ["NLL loss", "NLL loss", "Macro-F1"]):
    ax.set(title=title, xlabel="Epoch", ylabel=ylabel)
    ax.legend()
fig.tight_layout()
fig.savefig(results_folder / "repeated_seed_curves.png", dpi=150)
plt.show()

snellius_reference = None
if snellius_results is not None:
    settings_path = snellius_results / "settings.json"
    summary_path = snellius_results / "confirmation_summary.csv"
    if settings_path.exists() and summary_path.exists():
        snellius_settings = json.loads(settings_path.read_text())
        snellius_reference = pd.read_csv(summary_path)
        if snellius_settings.get("dataset_revision") != dataset_revision:
            raise ValueError("The separate Snellius study used a different dataset revision.")
        if snellius_reference.empty:
            raise ValueError("The separate Snellius summary is empty.")
        print("Verified separate Snellius files at:", snellius_results)
        print("Snellius dataset revision:", snellius_settings.get("dataset_revision"))
        display(snellius_reference)
    else:
        print("P2_SNELLIUS_RESULTS is set, but settings.json or confirmation_summary.csv is missing.")
else:
    print("No Snellius result path configured; rounded historical values above remain discussion only.")
print("Q5 answer: this measures training-seed variation on one fixed fold only.")
print("Further evaluation needs multiple held-out folds; a consistent pilot sign alone does not establish superiority.")

## Q6. Which text characteristics accompany errors?

**Features chosen before inspecting these results:** token length, OOV fraction,
negation and security terms. We now join these to held-out correctness and confidence
for each system. Error rates and counts by feature bin make the analysis testable.
These predictions come from a development fold used for comparisons, not an untouched test.
We show IDs rather than raw dataset prompts. Here `oov_fraction` means missing from
the actual GloVe-restricted model vocabulary, not necessarily from the full GloVe table.
The separate Snellius study also found some unknown-heavy errors with high confidence;
that is descriptive evidence, not a causal effect.

In [ ]:
# Q6a — join input features to actual held-out predictions
features = pd.DataFrame([input_features(text, tokenizer, prepared["vocab"])
                         for text in validation.prompt])
features["id"] = validation.id.to_numpy()
features["length_bin"] = pd.cut(features.token_length, [-1, 32, 128, np.inf],
                                  labels=["0–32", "33–128", "129+"])
features["oov_bin"] = pd.cut(features.oov_fraction, [-1, 0, 0.05, 1],
                               labels=["none", "0–5%", ">5%"])
errors = predictions[predictions.seed == seed].merge(features, on="id", validate="many_to_one")
errors["wrong"] = errors.gold != errors.prediction
errors["confidently_wrong"] = errors.wrong & errors.confidence.ge(0.8)
errors["false_negative"] = errors.gold.eq(1) & errors.prediction.eq(0)
errors["false_positive"] = errors.gold.eq(0) & errors.prediction.eq(1)
error_bins = []
major = errors[errors.system.isin(["mean_tuned", "lstm_tuned"])]
for feature in ["length_bin", "oov_bin", "has_negation", "has_security_term"]:
    table = major.groupby(["system", feature], observed=True).agg(
        n=("wrong", "size"), error_rate=("wrong", "mean"),
        harmful_share=("gold", "mean"), confidently_wrong=("confidently_wrong", "sum")).reset_index()
    table = table.rename(columns={feature: "bin"}).assign(feature=feature)
    error_bins.append(table)
    print(feature)
    display(table.round(4))
error_bins = pd.concat(error_bins, ignore_index=True)
display(major[major.wrong].sort_values("confidence", ascending=False)[
    ["system", "id", "gold", "prediction", "confidence", "token_length", "oov_fraction"]].head(10))
print("Q6 answer: the tables test the proposed associations, with group counts and class mix.")
print("Priorities: inspect enough errors in high-error groups, check negation/order and rare terms, then validate targeted changes on fresh folds.")
print("Small groups, correlated features and uncalibrated confidence limit interpretation; these are not causal effects.")

In [ ]:
# Save — evidence for the blog, with no raw prompt text
scores.to_csv(results_folder / "p2_scores.csv", index=False)
history_table.to_csv(results_folder / "p2_training_history.csv", index=False)
predictions.to_csv(results_folder / "p2_validation_predictions.csv", index=False)
features.to_csv(results_folder / "p2_input_features.csv", index=False)
error_bins.to_csv(results_folder / "p2_error_bins.csv", index=False)
variance.to_csv(results_folder / "p2_seed_variance.csv")
paired.to_csv(results_folder / "p2_paired_differences.csv")
q3_curves.to_csv(results_folder / "p2_mean_pretrained_vs_random_history.csv", index=False)
seed_curves.to_csv(results_folder / "p2_repeated_seed_curves.csv", index=False)
pd.DataFrame([p1_score]).to_csv(results_folder / "p1_matched_baseline_score.csv", index=False)
p1_predictions.to_csv(results_folder / "p1_matched_baseline_predictions.csv", index=False)
comparison.to_csv(results_folder / "p1_p2_matched_comparison.csv", index=False)
if snellius_reference is not None:
    snellius_reference.to_csv(results_folder / "snellius_separate_reference.csv", index=False)
settings["configurations"] = pool_configs + lstm_configs
settings["examples_per_class"] = examples_per_class
settings["p1_reference"] = {"system": "BPE + TF-IDF + logistic regression",
                            "vocab_size": 3000, "min_frequency": 2,
                            "C": 1.0, "solver": "lbfgs", "max_iter": 1000}
settings["snellius_reference"] = str(snellius_results) if snellius_reference is not None else None
settings["completed_fits"] = len(scores)
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print("Saved", len(scores), "fits and their evidence to", results_folder)
print("Next research step: freeze a shortlist and evaluate paired folds; keep the official test for final evaluation.")